# Memorias escritas en los pesos: ¿el modelo recuerda sin contexto? (Kaggle, T4)

Idea #9 del backlog ("living weights"). Modelo **Qwen3.5-0.8B** (fp32).

**Qué hace:** inventa personas que no existen, cada una con 3 datos (ciudad donde vive, trabajo, nombre de su perro). Escribe **una memoria por persona** en un `down_proj` (edición rango 1 estilo ROME: la clave es el último token del nombre). Solo se escriben 3 frases por persona; todo lo demás se mide con frases que el modelo nunca vio.

**Qué mide** (elección entre la respuesta correcta y 8 distractores, y "exacta" = greedy la produce):
- `para`: paráfrasis y preguntas en inglés (**el test real**; en GPT-2 quedó igual que la base)
- `es`: español · `chat`: con el template de chat · `hop`: el país de la ciudad (nunca escrito) · `reverse`: perro → dueño
- `nb_first` / `nb_last`: gente **no editada** con el mismo nombre / apellido (cuánto se filtra; más bajo es mejor)
- efectos colaterales: perplejidad WikiText-2, capitales que la base ya sabía, cuántos tokens disparan la compuerta
- brazos: `base`, `ctx` (los datos en el contexto: techo), `rome` (todas las memorias sumadas = 1 edición de pesos), `rome_gated` (cada memoria solo actúa donde su clave coincide), `memit` (una edición conjunta), `randkey` (control: claves al azar), `lastpos` (clave en el último token de la frase: lo que hacía GPT-2)
- escalas: 10, 100 y 1000 memorias, evaluadas siempre en las mismas 50 personas
- chequeo de reinicio: guarda el banco `.moimem`, recarga el modelo y el banco desde disco, mismas respuestas

**Uso:** Settings → **Accelerator: GPU T4** (x1 alcanza), **Internet: On** → **Save Version → Save & Run All (Commit)**. ~1.5–2 h. Resultado en el **Output**: `results_moi_memory.json` y `qwen35_0.8b_people_1000.moimem`.


In [ ]:
import os, json, subprocess, sys, shutil
REPO = "/kaggle/working/wsi"
BASE = "/tmp/qwen-base"
OUTD = "/kaggle/working"
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q -U transformers
!pip install -q flash-linear-attention || echo "fla opcional"
import transformers, torch; print("transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())


In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
assert os.path.exists(f"{REPO}/scripts/moi_memory.py"), "moi_memory.py no está en GitHub todavía: falta hacer push"
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3.5-0.8B", local_dir=BASE)
print("commit", COMMIT)


In [ ]:
# 1. barrido de capa (frases dev, 12 personas aparte) + memorias para 1000 personas + evaluación en 10/100/1000
OUT_TMP = "/tmp/results_moi_memory.json"
BANK = f"{OUTD}/qwen35_0.8b_people_1000.moimem"
!python -u {REPO}/scripts/moi_memory.py --model {BASE} --dtype fp32 --people 10,100,1000 --eval-people 50 --layers 3,5,7,9,11 --cov-tokens 100000 --ppl-tokens 32768 --tok-budget 8192 --save-bank {BANK} --out {OUT_TMP} 2>&1 | grep --line-buffered -v -E "Loading weights|fast path|Warning"


In [ ]:
# 2. resultado final (va al Output solo si terminó)
res = json.load(open(OUT_TMP))
assert res.get("complete"), "la corrida no terminó"
res["_final"] = {"complete": True, "commit": COMMIT, "notebook": "moi_memory_kaggle.ipynb", "gpu": torch.cuda.get_device_name(0)}
with open(f"{OUTD}/results_moi_memory.json.tmp", "w") as f:
    json.dump(res, f, indent=1, ensure_ascii=False)
os.replace(f"{OUTD}/results_moi_memory.json.tmp", f"{OUTD}/results_moi_memory.json")
shutil.rmtree(REPO, ignore_errors=True)    # el Output solo lleva los resultados y el banco
for N, sc in res["scales"].items():
    for arm, a in sc["arms"].items():
        p = a["probes"]
        print(N, arm.ljust(10), {k: p[k]["choice"] for k in ("write", "para", "es", "chat", "hop", "reverse", "nb_first", "nb_last") if k in p},
              "ppl", a.get("ppl", {}).get("ppl"), "caps", a.get("capitals_retained"))
print(sorted(os.listdir(OUTD)))
